# AR(1) grid: extra seeds {789, 1011} (Kaggle)

Trains the compound-symmetry AR(1) grid for seeds {789, 1011}, adding them to
the three original seeds {42, 123, 456} in `results/results_grid.csv`.
Runs CI, CD, and DLinear at C in {7, 21, 84}, rho in {0.1, 0.5, 0.9}: 54 runs,
about 4.2 h on a T4, one Kaggle session.

**Provenance.** The original three seeds were trained from series produced by
`src/generators/generate_ar1_grid.py` (13,400 usable timesteps). This notebook
carries its own inline generator, the code that actually ran for the extra
seeds: the same diagonal-transition AR(1) with compound-symmetry innovations,
but returning `T_TOTAL` = 14,400 rows after the burn-in discard and drawing
innovations through a Cholesky factor rather than `multivariate_normal`, so
the two implementations realise the same process from different random
streams. The paper's synthetic-protocol table records both usable lengths and
the resulting training-window counts. To retrain the original seeds from this
notebook set `NEW_SEEDS = [42, 123, 456]` and `T_TOTAL = 13_400`; the series
will follow the same process but not reproduce the committed rows bit for bit.

**Output:** `/kaggle/working/results_grid_extra_seeds.csv`, the same schema as
`results_grid.csv`; the committed file is the concatenation of the original
rows and this output with duplicate (C, rho, mode, seed) keys dropped.

Resume-safe: an atomic checkpoint is written after every run under
`/kaggle/working/`. The final cell compares the new seeds against the original
rows when `results_grid.csv` is attached as a Kaggle input dataset and is
skipped otherwise.


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ── Configuration (matches the original grid rows in results_grid.csv) ────────
PHI:  float = 0.8

C_VALUES:   list[int]   = [7, 21, 84]
RHO_VALUES: list[float] = [0.1, 0.5, 0.9]
MODES:      list[str]   = ["CI", "CD", "DLinear"]

# Seeds to train; the original grid seeds are {42, 123, 456}
NEW_SEEDS:  list[int]   = [789, 1011]

SEQ_LEN:      int = 512
PRED_LEN:     int = 96
PATCH_SIZE:   int = 16
PATCH_STRIDE: int = 8

D_MODEL:  int   = 64
N_HEADS:  int   = 8
N_LAYERS: int   = 3
DROPOUT:  float = 0.2

LR:             float = 1e-4
WEIGHT_DECAY:   float = 1e-4
MAX_EPOCHS:     int   = 50
PATIENCE:       int   = 10
GRAD_CLIP:      float = 1.0
WARMUP_EPOCHS:  int   = 10

# Batch sizes match the original grid rows, per the batch-size table in main.tex
BATCH_CI_BY_C:  dict[int,int] = {7: 128, 21: 128, 84: 32}
BATCH_CD_BY_C:  dict[int,int] = {7: 64,  21: 8,   84: 1}
BATCH_DL_BY_C:  dict[int,int] = {7: 128, 21: 128, 84: 32}

N_PATCHES = (SEQ_LEN - PATCH_SIZE) // PATCH_STRIDE + 1
print(f"N_PATCHES={N_PATCHES}  total new runs={len(C_VALUES)*len(RHO_VALUES)*len(MODES)*len(NEW_SEEDS)}")

T_TOTAL = 14_400   # usable rows after burn-in; the original seeds used 13_400
BURN_IN = 1_000
TRAIN_FRAC = 0.6
VAL_FRAC   = 0.2


In [ ]:
# ── AR(1) data generator ────────────────────────────────────────────────────
# T_TOTAL is the row count returned after the burn-in discard, not the count
# simulated: T_TOTAL + BURN_IN steps are simulated and X[BURN_IN:] returns
# exactly T_TOTAL rows, which the 60/20/20 split below divides 8,640/2,880/2,880.

def generate_ar1(C: int, phi: float, rho: float, seed: int) -> np.ndarray:
    rng   = np.random.default_rng(seed)
    Sigma = np.full((C, C), rho, dtype=np.float64)
    np.fill_diagonal(Sigma, 1.0)
    L     = np.linalg.cholesky(Sigma)
    total = T_TOTAL + BURN_IN
    X     = np.zeros((total, C), dtype=np.float64)
    for t in range(1, total):
        X[t] = phi * X[t-1] + L @ rng.standard_normal(C)
    return X[BURN_IN:]


def split_normalise(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_tr = int(len(data) * TRAIN_FRAC)
    n_va = int(len(data) * VAL_FRAC)
    tr, va, te = data[:n_tr], data[n_tr:n_tr+n_va], data[n_tr+n_va:]
    mean = tr.mean(axis=0, keepdims=True)
    std  = np.where(tr.std(axis=0, keepdims=True)==0, 1.0, tr.std(axis=0, keepdims=True))
    return (tr-mean)/std, (va-mean)/std, (te-mean)/std


def make_windows(data: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    T, Cv = data.shape
    n = T - SEQ_LEN - PRED_LEN + 1
    if n <= 0:
        raise ValueError(f"Not enough timesteps: {T}")
    s0, s1 = data.strides
    view = np.lib.stride_tricks.as_strided(
        data, shape=(n, SEQ_LEN+PRED_LEN, Cv), strides=(s0, s0, s1))
    xs = np.ascontiguousarray(view[:, :SEQ_LEN])
    ys = np.ascontiguousarray(view[:, SEQ_LEN:])
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)


def empirical_rho(train_data: np.ndarray) -> float:
    corr = np.corrcoef(train_data.T)
    i, j = np.triu_indices(corr.shape[0], k=1)
    return float(np.mean(corr[i, j]))


In [ ]:
# ── Models ────────────────────────────────────────────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.proj    = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.proj(x))


class PatchTST_CI(nn.Module):
    def __init__(self, C: int) -> None:
        super().__init__()
        self.C    = C
        self.embed = PatchEmbedding(PATCH_SIZE, D_MODEL, DROPOUT)
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_MODEL*4,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head    = nn.Linear(N_PATCHES * D_MODEL, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        x  = x.permute(0,2,1).reshape(B*Cv, L)
        p  = x.unfold(-1, PATCH_SIZE, PATCH_STRIDE)
        e  = self.encoder(self.embed(p))
        return self.head(e.reshape(B*Cv,-1)).reshape(B,Cv,-1).permute(0,2,1)


class PatchTST_CD(nn.Module):
    def __init__(self, C: int) -> None:
        super().__init__()
        self.C = C
        self.embed = PatchEmbedding(PATCH_SIZE, D_MODEL, DROPOUT)
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_MODEL*4,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head    = nn.Linear(N_PATCHES * D_MODEL, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        xp  = x.permute(0,2,1).reshape(B*Cv, L)
        p   = xp.unfold(-1, PATCH_SIZE, PATCH_STRIDE)
        emb = self.embed(p).reshape(B, Cv, N_PATCHES, -1)
        seq = emb.reshape(B, Cv*N_PATCHES, -1)
        enc = self.encoder(seq).reshape(B*Cv, -1)
        return self.head(enc).reshape(B,Cv,-1).permute(0,2,1)


class TrueDLinear(nn.Module):
    """Zeng et al. (2023): MA trend + remainder, two independent branches."""
    def __init__(self, seq_len: int) -> None:
        super().__init__()
        pad = (25 - 1) // 2
        self.avg_pool         = nn.AvgPool1d(kernel_size=25, stride=1, padding=pad)
        self.linear_trend     = nn.Linear(seq_len, PRED_LEN)
        self.linear_remainder = nn.Linear(seq_len, PRED_LEN)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, Cv = x.shape
        xf    = x.permute(0,2,1).reshape(B*Cv, 1, L)
        trend = self.avg_pool(xf).reshape(B*Cv, L)[:, :L]
        rem   = xf.reshape(B*Cv, L) - trend
        out   = self.linear_trend(trend) + self.linear_remainder(rem)
        return out.reshape(B, Cv, -1).permute(0,2,1)


def build_model(mode: str, C: int) -> nn.Module:
    if mode == "CI":      return PatchTST_CI(C)
    if mode == "CD":      return PatchTST_CD(C)
    if mode == "DLinear": return TrueDLinear(SEQ_LEN)
    raise ValueError(f"Unknown mode: {mode}")


def get_batch(mode: str, C: int) -> int:
    if mode == "CI":      return BATCH_CI_BY_C[C]
    if mode == "CD":      return BATCH_CD_BY_C[C]
    if mode == "DLinear": return BATCH_DL_BY_C[C]
    raise ValueError(mode)


# Assertions
for _C in C_VALUES:
    _cd = build_model("CD", _C)
    assert _cd.head.in_features == N_PATCHES * D_MODEL
    assert _cd.head.out_features == PRED_LEN
    _dl = build_model("DLinear", _C)
    assert _dl.linear_trend is not _dl.linear_remainder
    del _cd, _dl
free_cuda()
print("All model assertions passed.")


In [ ]:
# ── Training engine ───────────────────────────────────────────────────────────

def _cosine_warmup(opt, epoch: int, warmup: int) -> None:
    if epoch < warmup:
        lr = LR * (epoch + 1) / warmup
    else:
        p  = (epoch - warmup) / max(1, MAX_EPOCHS - warmup)
        lr = LR * 0.5 * (1.0 + np.cos(np.pi * p))
    for g in opt.param_groups:
        g["lr"] = lr


@torch.no_grad()
def _eval(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        pred = model(xb.to(DEVICE)).cpu()
        mse += nn.functional.mse_loss(pred, yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred,  yb, reduction="sum").item()
        n   += yb.numel()
    return mse/n, mae/n


def _fit(mode: str, C: int, rho: float, seed: int,
         datasets: tuple, batch_size: int, emp_rho: float) -> dict:
    x_tr,y_tr,x_va,y_va,x_te,y_te = datasets
    tr_dl = DataLoader(TensorDataset(x_tr,y_tr), batch_size=batch_size,
                       shuffle=True,  drop_last=False)
    va_dl = DataLoader(TensorDataset(x_va,y_va), batch_size=batch_size,
                       shuffle=False, drop_last=False)
    te_dl = DataLoader(TensorDataset(x_te,y_te), batch_size=batch_size,
                       shuffle=False, drop_last=False)

    use_warmup = mode in ("CI", "CD")
    model = build_model(mode, C).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda", enabled=(DEVICE.type=="cuda"))
    crit   = nn.MSELoss()

    spe = len(tr_dl)
    best_val = float("inf"); best_ep = 0; best_steps = 0
    best_state = None; no_imp = 0; total = 0

    try:
        for epoch in range(MAX_EPOCHS):
            if use_warmup:
                _cosine_warmup(opt, epoch, WARMUP_EPOCHS)
            model.train()
            for xb,yb in tr_dl:
                opt.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(DEVICE.type=="cuda")):
                    loss = crit(model(xb.to(DEVICE)), yb.to(DEVICE))
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update()
                total += 1
            val_mse, _ = _eval(model, va_dl)
            if val_mse < best_val:
                best_val=val_mse; best_ep=epoch+1; best_steps=total
                best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
                no_imp=0
            else:
                no_imp+=1
                if no_imp >= PATIENCE:
                    break
    except Exception:
        # No evaluation on the failure path: a second forward pass after an
        # OOM can raise a secondary exception that replaces the original one,
        # which train_one inspects to decide whether to retry.
        del model, opt, scaler
        free_cuda()
        raise
    else:
        if best_state:
            model.load_state_dict(best_state)
        test_mse, test_mae = _eval(model, te_dl)
        del model, opt, scaler
        free_cuda()

    return {"dataset":"synthetic_ar1","C":C,"rho":rho,"empirical_rho":emp_rho,
            "mode":mode,"pred_len":PRED_LEN,"seed":seed,
            "test_mse":test_mse,"test_mae":test_mae,"best_epoch":best_ep,
            "batch_size":batch_size,"steps_per_epoch":spe,"total_steps":best_steps}


def train_one(mode: str, C: int, rho: float, seed: int) -> dict:
    set_seed(seed)
    raw = generate_ar1(C, PHI, rho, seed)
    tr, va, te = split_normalise(raw)
    emp = empirical_rho(tr)
    datasets = (*make_windows(tr), *make_windows(va), *make_windows(te))
    batch_size = get_batch(mode, C)
    while True:
        try:
            set_seed(seed)
            return _fit(mode, C, rho, seed, datasets, batch_size, emp)
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            free_cuda()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size//2)
            print(f"  OOM: retrying batch_size={batch_size}")


In [ ]:
# ── Main sweep ────────────────────────────────────────────────────────────────
OUT = Path("/kaggle/working/results_grid_extra_seeds.csv")
TMP = OUT.with_suffix(".csv.tmp")


def _key(C: int, rho: float, mode: str, seed: int) -> tuple:
    return (int(C), round(float(rho), 4), str(mode), int(seed))


def _save(rows: list) -> None:
    pd.DataFrame(rows).to_csv(TMP, index=False)
    os.replace(TMP, OUT)
    assert OUT.exists() and OUT.stat().st_size > 100, f"checkpoint not written to {OUT}"


if OUT.exists() and OUT.stat().st_size > 100:
    _ex   = pd.read_csv(OUT)
    done  = {_key(r.C, r.rho, r['mode'], r.seed) for r in _ex.itertuples()}
    results = _ex.to_dict("records")
    print(f"Resuming: {len(done)} runs already done.")
else:
    done, results = set(), []

total = len(C_VALUES) * len(RHO_VALUES) * len(MODES) * len(NEW_SEEDS)
idx, fails = 0, []

for C in C_VALUES:
    for rho in RHO_VALUES:
        for mode in MODES:
            for seed in NEW_SEEDS:
                idx += 1
                key = _key(C, rho, mode, seed)
                if key in done:
                    print(f"[{idx}/{total}] SKIP C={C} rho={rho} mode={mode} seed={seed}")
                    continue
                print(f"[{idx}/{total}] C={C} rho={rho} mode={mode} seed={seed} ...",
                      end=" ", flush=True)
                t0 = time.time()
                try:
                    row = train_one(mode, C, rho, seed)
                except Exception as exc:
                    free_cuda()
                    print(f"FAILED: {type(exc).__name__}: {exc}")
                    fails.append((C, rho, mode, seed, repr(exc)))
                    continue
                print(f"mse={row['test_mse']:.4f}  epoch={row['best_epoch']}"
                      f"  bs={row['batch_size']}  ({time.time()-t0:.0f}s)")
                results.append(row)
                done.add(key)
                _save(results)

print(f"\nDone. {len(results)}/{total} runs -> {OUT}")
if fails:
    print(f"{len(fails)} failures (rerun this cell to retry):")
    for f in fails:
        print(" ", f)

In [ ]:
# ── Comparison against the original seeds at the same (C, rho, mode) ─────────
df_new = pd.read_csv(OUT)

candidates = [
    "/kaggle/working/results_grid.csv",
    *[str(p) for p in Path("/kaggle/input").glob("**/results_grid.csv")],
]
original = next((c for c in candidates if Path(c).exists()), None)

if original is None:
    print("results_grid.csv not found; skipping comparison.")
    print("Searched:", *candidates, sep="\n  ")
else:
    df_old = pd.read_csv(original)
    df_old = df_old[~df_old["seed"].isin(NEW_SEEDS)]  # the committed file already holds the new seeds
    print(f"Loaded original rows: {original}")
    print("=== Mean MSE comparison: original seeds vs new seeds ===")
    for mode in ["CI", "CD", "DLinear"]:
        old = df_old[df_old["mode"] == mode].groupby(["C", "rho"])["test_mse"].mean()
        new = df_new[df_new["mode"] == mode].groupby(["C", "rho"])["test_mse"].mean()
        diff = (new - old).abs()
        if diff.empty:
            print(f"  {mode}: no overlapping (C, rho) cells to compare")
            continue
        worst = diff.idxmax()
        print(f"  {mode}: max |new-old mean| = {diff.max():.4f} at C,rho={worst}  "
              f"(expect < 0.05; high-variance cells such as C=21, rho=0.9 may exceed)")

print("\n=== New seed distribution ===")
print(df_new.groupby(["C", "mode"])["test_mse"]
      .agg(["mean", "std", "count"]).round(4).to_string())

In [ ]:
from IPython.display import FileLink
FileLink(str(OUT))